# 20 — Download & Verifikasi Data (MOT20 + DanceTrack)

Kernel: `s2-main`.

| Dataset | Sumber | Catatan |
|---|---|---|
| MOT20 | Kaggle `ismailelbouknify/mot-20` → fallback HF `Lekim89/MOT20` → resmi `motchallenge.net` | **Wajib** `verify_mot_dataset.py`: banyak mirror Kaggle buang track ID |
| DanceTrack | HF `noahcao/dancetrack` (resmi; cc-by-4.0) | unduh tanpa `test/*` |

Lisensi: MOT20 riset-only (motchallenge.net), DanceTrack cc-by-4.0 non-komersial. Jangan
distribusikan ulang data mentah (folder `data/` sudah di-gitignore).

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

### 2.1 MOT20 — coba Kaggle dulu

In [ ]:
# butuh ~/.kaggle/kaggle.json (lihat https://www.kaggle.com/docs/api)
!pip show kaggle >/dev/null 2>&1 && echo "kaggle CLI ada" || echo "kaggle CLI belum ada"
!test -f ~/.kaggle/kaggle.json && echo "kaggle.json ada" || echo "kaggle.json TIDAK ada — upload dulu atau pakai fallback HF di bawah"

In [ ]:
!kaggle datasets download -d ismailelbouknify/mot-20 -p $S2_DATA/mot20_raw

In [ ]:
!mkdir -p $S2_DATA/mot20_raw/extracted && unzip -q $S2_DATA/mot20_raw/mot-20.zip -d $S2_DATA/mot20_raw/extracted
!find $S2_DATA/mot20_raw/extracted -maxdepth 3 -type d | head -30

**Jika Kaggle gagal** (login/404): jalankan sel fallback ini (HF mirror) lalu lanjut.

### Token HF (biar tidak kena rate limit 429)

Download anonim di-rate-limit per IP, dan repo MOT20 (`Lekim89/MOT20`)
menyimpan ~13 ribu file (gambar per-frame) — dengan Xet Storage aktif HF
memanggil endpoint API per file (kuota 1000 request/5 menit → pasti 429).
Sel token di bawah set `HF_HUB_DISABLE_XET=1` supaya download lewat CDN
langsung (tidak dihitung kuota API).

Buat **Read token** gratis:
https://huggingface.co/settings/tokens (New token → type **Read**).

Cara termudah: di terminal jalankan `huggingface-cli login` lalu paste token
(tersimpan di cache, semua sel download otomatis terbaca). Alternatif: isi
nilai `HF_TOKEN` di sel di bawah ini, lalu jalankan ulang sel download
(download resume otomatis — file yang sudah masuk tidak diulang).


In [ ]:
import os
# set token SEKALI; tidak menimpa yang sudah terpasang
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = "hf_xxx"   # ganti dengan token Read kamu
# repo MOT20 = 13k file; Xet memakai endpoint API per file -> kena kuota 1000/5mnt.
# Disable Xet = download CDN langsung (bukan API) -> tidak kena kuota.
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("HF_TOKEN set:", os.environ["HF_TOKEN"][:8] + "...")
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))


In [ ]:
# FALLBACK: HF Lekim89/MOT20
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Lekim89/MOT20", repo_type="dataset", local_dir=str(DATA/"mot20_hf"), ignore_patterns=["test/*"])

### 2.2 Deteksi & selaraskan ke layout kerja

Layout yang dicari: `{seq}/img1/*.jpg` + `{seq}/gt/gt.txt` + `{seq}/seqinfo.ini`.
Sel berikut memindai semua kandidat sekuens di bawah `$S2_DATA` — tidak mengasumsikan
struktur mirror.

In [ ]:
import glob
cands = []
for gt in glob.glob(str(DATA / "**" / "gt" / "gt.txt"), recursive=True):
    seq_dir = Path(gt).parent.parent
    if (seq_dir / "img1").exists():
        n = len(list((seq_dir / "img1").glob("*.*")))
        cands.append((seq_dir.name, str(seq_dir), n))
for name, p, n in cands:
    print(f"{name:16s} frames={n:6d}  {p}")

Review daftar di atas. MOT20 yang valid = 4 sekuens TRAIN: `MOT20-01, MOT20-02, MOT20-03, MOT20-05`
(test 04/06/07/08 tidak punya GT publik). Sesuaikan `SRC_MOT20` bila perlu, lalu symlink.

In [ ]:
import os
SRC_MOT20 = Path(input("Path folder berisi sekuens MOT20 train (lihat list di atas): ").strip())
TRAIN_NAMES = ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]
dst = DATA / "mot20" / "train"; dst.mkdir(parents=True, exist_ok=True)
for name in TRAIN_NAMES:
    s = SRC_MOT20 / name
    if s.exists():
        (dst / name).symlink_to(s, target_is_directory=True)
        print("link", name)
    else:
        print("!! tidak ada:", s)

### 2.3 Verify MOT20 — WAJIB lulus

In [ ]:
!python $S2_ROOT/scripts/data_prep/verify_mot_dataset.py $S2_DATA/mot20/train

**Jika gagal** (mis. kolom ID hilang — mirror terkonversi jadi dataset deteksi):
ganti sumber (HF/resmi) dan ulangi dari 2.1/2.2. Tanpa GT ber-ID, HOTA/IDF1/IDSW tidak bisa dihitung.

### 2.4 DanceTrack — HF resmi, tanpa split test

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="noahcao/dancetrack", repo_type="dataset",
                  local_dir=str(DATA / "dancetrack_hf"), ignore_patterns=["test/*", "test*", "train*", "*.xlsx"])

In [ ]:
import os, zipfile
from pathlib import Path
src_val = DATA / "dancetrack_hf" / "val"
if not src_val.is_dir():
    z = DATA / "dancetrack_hf" / "val.zip"
    if z.exists():
        print('extract', z)
        src_val.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(src_val)
    else:
        # beberapa mirror menaruh sekuens langsung di bawah dancetrack_hf/
        src_val = DATA / "dancetrack_hf"
dst = DATA / "dancetrack" / "val"; dst.mkdir(parents=True, exist_ok=True)
# cari semua folder berisi img1 (tahan terhadap zip yang punya folder pembungkus)
seqs = sorted(p.parent for p in src_val.rglob("img1") if any(p.parent.iterdir()))
n = 0
for s in seqs:
    t = dst / s.name
    if t.exists():
        continue
    t.symlink_to(s, target_is_directory=True)
    n += 1
print(f'{len(seqs)} sekuens val ditemukan, {n} ditautkan baru')


In [ ]:
!python $S2_ROOT/scripts/data_prep/verify_mot_dataset.py $S2_DATA/dancetrack/val --min-sequences 20

### 2.5 seqinfo.ini — synthesize bila hilang

DiffMOT (`info_dir`) dan TrackEval butuh `seqinfo.ini` per sekuens (imWidth/imHeight/imExt/seqLength).

In [ ]:
import cv2, glob
from pathlib import Path
def synth_seqinfo(seq_dir: Path):
    ini = seq_dir / "seqinfo.ini"
    if ini.exists():
        return False
    imgs = sorted((seq_dir / "img1").glob("*.*"))
    if not imgs:
        print("!! tidak ada img1:", seq_dir); return False
    h, w = cv2.imread(str(imgs[0])).shape[:2]
    ext = imgs[0].suffix
    ini.write_text(
        f"[Sequence]\nname={seq_dir.name}\nimDir=img1\nframeRate=30\n"
        f"seqLength={len(imgs)}\nimWidth={w}\nimHeight={h}\nimExt={ext}\n"
    )
    print("synthesize", seq_dir.name, f"{w}x{h} x{len(imgs)}"); return True

n = 0
for seq_dir in sorted(DATA.glob("mot20/train/*")) + sorted(DATA.glob("dancetrack/val/*")):
    if seq_dir.is_dir():
        n += synth_seqinfo(seq_dir)
print("total synthesize:", n)

### 2.6 Ringkasan

In [ ]:
for split_root in [DATA/"mot20"/"train", DATA/"dancetrack"/"val"]:
    if not split_root.exists():
        print("!! belum ada:", split_root); continue
    for seq in sorted(p for p in split_root.iterdir() if p.is_dir()):
        n = len(list((seq/"img1").glob("*.*")))
        has_gt = (seq/"gt"/"gt.txt").exists()
        has_ini = (seq/"seqinfo.ini").exists()
        print(f"{seq.name:16s} frames={n:6d} gt={has_gt} seqinfo={has_ini}")

**Lanjut**: `30_s2_gen_detections.ipynb` — generate deteksi dengan bobot fine-tune Skenario A.